In [ ]:
import medal

from captum.attr import DeepLift
from captum.attr import FeatureAblation
from captum.attr import IntegratedGradients
from captum.attr import Occlusion
from captum.attr import KernelShap



import pickle
import matplotlib.pyplot as plt
import torch
import numpy as np
from tqdm import tqdm
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

In [3]:
#retrieve MNIST data
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
X_scaled=X/255.0

train=X_scaled[:10000]
labels_train=y[:10000]

test=X_scaled[10000:10500]
labels_test=y[10000:10500]

In [33]:
###functions for plotting feature attribution values for MNIST dataset

def overlay_features(fa_values, title, top_k=10):
    '''
    This function colors the top k pixels with the highest magnitude of 
    feature attribution values on top of the grayscaled image for each digit class. 
    This function shows local attribution.
    
    Input: 
        fa_values: a list of 10 lists of local feature attribution values for each digit class 
        title: str; title for the figure
        top_k: int; how many pixels to color
    Output:
        Displays a 2 by 5 plot of gray scaled digits 0-9 with colored pixels.
    '''
    
    fig, ax=plt.subplots(2,5,figsize=(10,5))
    for i in range(10):   #from digits 0-9
        fa_val=fa_values[i][0]   #on the first image 
        og_image=train[labels_train==str(i)][0].reshape(28,28) #on the first image

        top_indices=np.argsort(np.abs(fa_val.cpu().numpy()), axis=None)[-top_k:]
        top_y,top_x=np.unravel_index(top_indices,(28,28))

        colors=np.zeros((28,28,3), dtype=float)
        for y,x in zip(top_y,top_x):
            if fa_val[y*28+x].item()>0:
                colors[y,x]=[1,0,0]  #red if attribution value is positive
            else:
                colors[y,x]=[0,0,1]  #blue if attribution value is negative 
        ax[i//5, i%5].imshow(og_image, cmap="gray")
        ax[i//5, i%5].imshow(colors, alpha=0.6)
        ax[i//5, i%5].axis("off")
    fig.suptitle(title)

def global_overlay_features(fa_avg_values, title, top_k=10):
    '''
    This function colors the top k pixels with the highest magnitude of 
    feature attribution values on top of the averaged grayscaled image of each digit class. 
    This function shows global attribution.
    
    Input: 
        fa_avg_values: a list of 10 lists of averaged feature attribution values for each digit class 
        title: str; title for the figure
        top_k: int; how many pixels to color
    Output:
        Displays a 5 by 2 plot of gray scaled digits 0-9 with colored pixels.
    '''
    fig, ax=plt.subplots(2,5,figsize=(10,5))
    for i in range(10):
        fa_avg_val=fa_avg_values[i]
        og_image=train[labels_train==str(i)].mean(axis=0).reshape(28,28)

        top_indices=np.argsort(np.abs(fa_avg_val.detach().cpu().numpy()), axis=None)[-top_k:]
        top_y,top_x=np.unravel_index(top_indices,(28,28))

        colors=np.zeros((28,28,3), dtype=float)
        for y,x in zip(top_y,top_x):
            if fa_avg_val[y*28+x].item()>0:
                colors[y,x]=[1,0,0]  #red if avg attr value is positive
            else:
                colors[y,x]=[0,0,1]  #blue if avg attr value is negative
        ax[i//5, i%5].imshow(og_image, cmap="gray")
        ax[i//5, i%5].imshow(colors, alpha=0.6)
        ax[i//5, i%5].axis("off")
    fig.suptitle(title)

def heatmap_features(fa_values_avg, title):
    '''
    This function plots a heatmap of the global feature attribution values
    Input: 
        fa_values: a list of 10 lists of averaged feature attribution values for each digit class
        title: str; title for the figure
    Output:
        Displays a 5 by 2 heatmap plot of global feature attribution values.
    '''
    fig, ax=plt.subplots(2,5,figsize=(10,5))
    for i in range(10):
        fa_val=fa_values_avg[i]/torch.sum(fa_values_avg[i])
        
        ax[i//5, i%5].imshow(fa_val.detach().cpu().numpy().reshape(28,28),cmap="turbo")
        ax[i//5, i%5].axis("off")
    fig.suptitle(title)


In [5]:
with open(f"MEDAL-o/model_mnist_tsne.pkl",'rb') as file:
    model=pickle.load(file)

In [ ]:
#Deep lift

dl_values_list_avg=[]
dl_values_list_l2=[]

ds=DeepLift(model.model.encoder,
            multiply_by_inputs=True) #if true then global

for i in tqdm(range(10)):
    attr0,_=ds.attribute(torch.tensor(train[labels_train==str(i)]).float().to('cuda'),
                  target=0,
                  return_convergence_delta=True,
                  )
    attr1,_=ds.attribute(torch.tensor(train[labels_train==str(i)]).float().to('cuda'),
                  target=1,
                  return_convergence_delta=True,
                  )

    dl_values_list_avg.append(torch.mean((attr0+attr1)/2, dim=0))
    
    l2=torch.sqrt(torch.pow(attr0,2) + torch.pow(attr1,2))
    dl_values_list_l2.append(torch.mean(l2, dim=0))

heatmap_features(dl_values_list_l2,"Deep lift normalized L2")

In [ ]:
#Feature ablation
fa_values_list_avg=[]   
fa_values_list_l2=[]

abl=FeatureAblation(model.model.encoder)

for i in range(10):
    attr0=abl.attribute(torch.tensor(train[labels_train==str(i)]).float().to('cuda'),
                  target=0,
                  n_steps=200)
    attr1=abl.attribute(torch.tensor(train[labels_train==str(i)]).float().to('cuda'),
                  target=1,
                  n_steps=200)

    fa_values_list_avg.append(torch.mean((attr0+attr1)/2, dim=0))
    
    l2=torch.sqrt(torch.pow(attr0,2) + torch.pow(attr1,2))
    fa_values_list_l2.append(torch.mean(l2, dim=0))

heatmap_features(fa_values_list_l2,"Feature ablation normalized L2")

In [ ]:
#Integrated gradients
ig_values_list_avg=[]
ig_values_list_l2=[]

ig=IntegratedGradients(model.model.encoder)

for i in tqdm(range(10)):
    attr0,_=ig.attribute(torch.tensor(train[labels_train==str(i)]).float().to('cuda'),
                  target=0,
                  return_convergence_delta=True,
                  n_steps=200)
    attr1,_=ig.attribute(torch.tensor(train[labels_train==str(i)]).float().to('cuda'),
                  target=1,
                  return_convergence_delta=True,
                  n_steps=200)

    ig_values_list_avg.append(torch.mean((attr0+attr1)/2, dim=0))
    
    l2=torch.sqrt(torch.pow(attr0,2) + torch.pow(attr1,2))
    ig_values_list_l2.append(torch.mean(l2, dim=0))

heatmap_features(ig_values_list_l2,"Integrated gradients normalized L2")

In [ ]:
#Occlusion
oc_values_list_avg=[]
oc_values_list_l2=[]

oc=Occlusion(model.model.encoder) 

for i in tqdm(range(10):
    sub_train=torch.tensor(train[labels_train==str(i)]).float().to('cuda')
    attr0=oc.attribute(sub_train,
                  sliding_window_shapes=(5,),
                  target=0,
                  )
    attr1=oc.attribute(sub_train,
                  target=1,
                  sliding_window_shapes=(5,)
                  )

    oc_values_list_avg.append(torch.mean((attr0+attr1)/2, dim=0))
    
    l2=torch.sqrt(torch.pow(attr0,2) + torch.pow(attr1,2))
    oc_values_list_l2.append(torch.mean(l2, dim=0))

heatmap_features(oc_values_list_l2,"Occlusion normalized L2")

In [ ]:
#Kernel SHAP
ds_values_list_avg=[]
ds_values_list_l2=[]

dl=KernelShap(model.model.encoder)

for i in tqdm(range(10):
    attr0=dl.attribute(torch.tensor(train[labels_train==str(i)]).float().to('cuda'),
                  target=0,
                  show_progress=False
                  )
    attr1=dl.attribute(torch.tensor(train[labels_train==str(i)]).float().to('cuda'),
                  target=1,
                  show_progress=False
                  )
    
    ds_values_list_avg.append(torch.mean((attr0+attr1)/2, dim=0))
    
    l2=torch.sqrt(torch.pow(attr0,2) + torch.pow(attr1,2))
    ds_values_list_l2.append(torch.mean(l2, dim=0))

heatmap_features(ds_values_list_l2,"Kernel shap normalized L2")